In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys

print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.HelperFunctions import HelperFunctions

from makedf.mcstat import get_MCstat_unc

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/syst/detector"

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
# BNB data
print("data_tot_pot: %.3e" %(data_tot_pot))
print("data tot gates : %.3e" %(data_gates))

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from functools import partial

pot_weight_col = ('slc', 'wgt', '', '', '', '')

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

# --- Configuration & Setups ---
syst_keys = ["SystVarsCV","wiremod_YZ","wiremod_XZ_thetaXW", "0xSCE","2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]
colors = ["black", "C0","C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14","C15","C16"]
labels = ["CV", r"Wiremod Y-Z",r"Wiremod X$\theta_{xw}$", "0xSCE", "2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]

selection_string = "_ar23p"

keys2load = ["cc1pi", "hdr", "nudf"] 
file_dir = "/exp/sbnd/data/users/lpelegri/syst/CCBC_rates" + selection_string
os.makedirs(file_dir, exist_ok=True)

unisim_keys = []
paired_syst = {} 

# Sort Unisim vs Paired/Multisim
for key in syst_keys:
    if key.endswith('m') or key.endswith('p'):
        base = key[:-1]
        if base not in paired_syst: paired_syst[base] = [None, None]
        if key.endswith('m'): paired_syst[base][0] = key
        else: paired_syst[base][1] = key
    else:
        unisim_keys.append(key)

# Intermediate flat arrays to preserve variable metrics cleanly
intermediate_hists = {key: {"Ps": {}, "Bs": {}, "nc": {}} for key in syst_keys}

cols_to_keep_mc = [
    ('nu_categ', '', '', ''),
    ('nu_categ_proton_reduced', '', '', ''),
    ('genie_categ', '', '', ''),
    ('genie_mode', '', '', ''),
]

detvar_plotter_final_vars = partial(
    variation_hists_final_vars,
    var_colors=colors,
    var_labels=labels,
    approval="internal"
)

print(f"📋 Configuration initialized. Total variables to process: {len(var_configs)}")
print(f"📊 Unisim keys ({len(unisim_keys)}): {unisim_keys}")
print(f"📊 Paired variations ({len(paired_syst)}): {list(paired_syst.keys())}")

# ==============================================================================
# PIPELINE: SYSTEMATIC-BY-SYSTEMATIC PROCESSING (1 Disk Read per File)
# ==============================================================================
for idx, key in enumerate(syst_keys, 1):
    print(f"\n{'='*80}")
    print(f"💿 [{idx}/{len(syst_keys)}] Loading & Processing Systematic File: {key}")
    print(f"{'='*80}")
    
    # 1. Read file from disk EXACTLY ONCE per systematic run
    temp_df = load_df(f"/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_{key}.df", keys2load, 1000, filter_df = False)
    print(f"   ↳ Loaded 'cc1pi' shape: {temp_df['cc1pi'].shape} | 'nudf' shape: {temp_df['nudf'].shape}")
    
    # 2. Scale POT
    syst_tot_pot = temp_df['hdr']['pot'].sum()
    syst_pot_scale = data_tot_pot / syst_tot_pot
    temp_df['cc1pi'][pot_weight_col] = syst_pot_scale
    print(f"   ↳ POT Scale Factor: {syst_pot_scale:.4e} (Total POT: {syst_tot_pot:.4e})")

    print("   ↳ Starting Truth Matching...")
    # 3. Truth Match
    temp_df['nudf'] = temp_df['nudf'][cols_to_keep_mc]
    matched_evt = perform_truth_matching(temp_df['cc1pi'], temp_df['nudf'])
    print(f"   ↳ Finished Truth Matching. Matched shape: {matched_evt.shape}")
    
    # 4. Filter via Masks
    print("   ↳ Evaluating event selection masks...")
    normal_masks = build_event_cumulative_masks(matched_evt, sideband="")["energy"]
    proton_mask = build_event_cumulative_masks(matched_evt, sideband="proton")["energy"]
    
    df_normal = matched_evt[normal_masks]
    df_sideband = matched_evt[proton_mask]
    print(f"   ↳ Mask retention -> Normal: {len(df_normal)} rows | Proton Sideband: {len(df_sideband)} rows")
    
    # 5. Extract Signal & Background Event Subsets
    df_sig = df_normal[df_normal.truth.nu_categ == "CC1pi"].groupby(level=['__ntuple', 'entry', 'rec.slc..index'], sort=False).first()
    df_bkg = df_normal[df_normal.truth.nu_categ != "CC1pi"].groupby(level=['__ntuple', 'entry', 'rec.slc..index'], sort=False).first()
    df_sb  = df_sideband.groupby(level=['__ntuple', 'entry', 'rec.slc..index'], sort=False).first()
    print(f"   ↳ Groupby collapsed slices -> Signal: {len(df_sig)} | Background: {len(df_bkg)} | Sideband: {len(df_sb)}")
    
    # Clean up intermediate DataFrames no longer required
    del temp_df, matched_evt, df_normal, df_sideband
    gc.collect()

    # 6. Inner Loop: Extract 1D Histograms for All Variables Sequentially
    print("   ↳ Extracting histograms:")
    for v_idx, var_config in enumerate(var_configs, 1):
        var_name = var_config.var_save_name
        var_column = var_config.var_evt_reco_col
        print(f"      [{v_idx}/{len(var_configs)}] -> {var_name}")
        
        # Calculate tiny 1D array slices
        hist_sig = detvar_plotter_final_vars([df_sig], var_name=var_column, bins=var_config.bins, plot=False)[0]
        hist_bkg = detvar_plotter_final_vars([df_bkg], var_name=var_column, bins=var_config.bins, plot=False)[0]
        hist_sb  = detvar_plotter_final_vars([df_sb],  var_name=var_column, bins=var_config.bins, plot=False)[0]
        
        # Save tiny lightweight arrays inside storage container
        intermediate_hists[key]["Ps"][var_name] = hist_sig
        intermediate_hists[key]["Bs"][var_name] = hist_bkg
        intermediate_hists[key]["nc"][var_name] = hist_sb

    # 7. Complete Evacuation of Large Data structures from RAM before handling next systematic file
    del df_sig, df_bkg, df_sb
    gc.collect()
    print(f"   ↳ Cleared temporary variables from RAM for {key}.")


# ==============================================================================
# POST-PROCESSING: PACKING STRUCT FOR MULTISIM FORMATS
# ==============================================================================
print("\n📦 Re-structuring universe tracking maps...")
univ_hist_detector = {}

# Populate Unisims (Shape: 1, nbins)
print("   ↳ Packing Unisim structures...")
for syst_key in unisim_keys:
    univ_hist_detector[syst_key] = {"Ps": {}, "Bs": {}, "nc": {}}
    for var_config in var_configs:
        v_name = var_config.var_save_name
        univ_hist_detector[syst_key]["Ps"][v_name] = np.array(intermediate_hists[syst_key]["Ps"][v_name], ndmin=2)
        univ_hist_detector[syst_key]["Bs"][v_name] = np.array(intermediate_hists[syst_key]["Bs"][v_name], ndmin=2)
        univ_hist_detector[syst_key]["nc"][v_name] = np.array(intermediate_hists[syst_key]["nc"][v_name], ndmin=2)
         
# Populate Paired sets (Shape: 2, nbins)
print("   ↳ Packing Paired multisim structures...")
for base_name, keys in paired_syst.items():
    univ_hist_detector[base_name] = {"Ps": {}, "Bs": {}, "nc": {}}
    key_m, key_p = keys[0], keys[1]
    
    if key_m in intermediate_hists and key_p in intermediate_hists:
        for var_config in var_configs:
            v_name = var_config.var_save_name
            univ_hist_detector[base_name]["Ps"][v_name] = np.stack([intermediate_hists[key_m]["Ps"][v_name], intermediate_hists[key_p]["Ps"][v_name]])
            univ_hist_detector[base_name]["Bs"][v_name] = np.stack([intermediate_hists[key_m]["Bs"][v_name], intermediate_hists[key_p]["Bs"][v_name]])
            univ_hist_detector[base_name]["nc"][v_name] = np.stack([intermediate_hists[key_m]["nc"][v_name], intermediate_hists[key_p]["nc"][v_name]])


# ==============================================================================
# DISK SAVE
# ==============================================================================
save_path = os.path.join(file_dir, "detector_univ_hists.npz")
print(f"\n💾 Writing final array dictionary payload to: {save_path}")
np.savez(save_path, **{k: np.array(v, dtype=object) for k, v in univ_hist_detector.items()})
print("🏁 Process complete. All files saved seamlessly!")